# Debugging with Logging: Practice Exercise

In the guided practice, you learned how to add basic structured logging to a ReAct agent. Now you'll extend that logging with production-ready features.

**What you'll implement:**
- Tool execution time tracking (`duration_ms`)
- Summary statistics at query completion
- Error logging when max turns is exceeded

**Estimated time:** 10-15 minutes

In [1]:
from openai import OpenAI
import re
import logging
import json
import time
from datetime import datetime, timezone
import os

In [2]:
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_TPU_ADDR' in os.environ


if not IN_COLAB:
  from dotenv import load_dotenv
  load_dotenv()
  # Verify OpenAI API key is set
  if not os.getenv("OPENAI_API_KEY"):
    ("WARNING: OPENAI_API_KEY environment variable not set!")
  else:
    OPEN_AI_API_KEY=os.getenv("OPENAI_API_KEY")
    print("OpenAI API key found. Ready to proceed!")
else:
  from google.colab import userdata
  OPEN_AI_API_KEY=userdata.get('OPEN_AI_API_KEY')

## Setup

Run this cell to import dependencies and configure the logging system.

In [3]:
from openai import OpenAI
import re
import logging
import json
import time
from datetime import datetime, timezone
import os

# Initialize OpenAI client
client = OpenAI(api_key=OPEN_AI_API_KEY)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    force=True # Ensure the configuration is applied
)
logger = logging.getLogger(__name__)

print("Setup complete.")

Setup complete.


## Provided Code

The `log_event` helper, Agent class, tools, and system prompt are provided from the guided practice. Review the `log_event` function - you'll use it to add new logging calls.

In [4]:
def log_event(event_type: str, data: dict) -> None:
    """Log an event with structured data in JSON format"""
    log_entry ={
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "event": event_type,
        "data":data
    }
    logger.info(json.dumps(log_entry))

In [5]:
log_event("test message",{"test_key": "test_value"})

2026-03-28 02:30:37 - INFO - {"timestamp": "2026-03-28T02:30:37.809427+00:00", "event": "test message", "data": {"test_key": "test_value"}}


In [9]:
class Agent:
    """Agent class that handles OpenAI API interactions."""

    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        log_event("message_added", {"role": "user", "content_preview": message[:100] + "..." if len(message) > 100 else message})
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        log_event("message_added", {"role": "assistant", "content_preview": result[:100] + "..." if len(result) > 100 else result })
        return result

    def execute(self):

        log_event("llm_input", {"message_count": len(self.messages), "last_message_preview": str(self.messages[-1])[:200]})
        completion = client.chat.completions.create(
            model="gpt-4o",
            temperature=0,
            messages=self.messages
        )
        result = completion.choices[0].message.content
        log_event("llm_output", {
            "content_length": len(result),
            "completion_preview": result[:200] + "..." if len(result) >200 else result,
            "tokens": { "prompt": completion.usage.prompt_tokens,
                       "completion": completion.usage.completion_tokens,
                        "total": completion.usage.total_tokens,
            },
          "model": completion.model,
            })
        return result


# Fruit pricing tools
def get_fruit_price(fruit):
    """Get the price of a specific fruit."""
    time.sleep(0.05)  # Simulate processing time
    prices = {
        "apple": 1.5,
        "banana": 1.2,
        "orange": 1.3,
        "grape": 2.0
    }
    return prices.get(fruit.lower(), "Unknown fruit")


def calculate_total_price(fruits_str):
    """Calculate total price for multiple fruits."""
    time.sleep(0.03)  # Simulate processing time
    items = fruits_str.split(',')
    total = 0
    for item in items:
        parts = item.strip().split()
        quantity = int(parts[0])
        fruit = parts[1].rstrip('s')  # Remove plural 's'
        price = get_fruit_price(fruit)
        if isinstance(price, str):
            return price
        total += quantity * price
    return total


# System prompt for the agent
system_prompt = """
You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

get_fruit_price:
e.g. get_fruit_price: apple
Returns the price of a fruit

calculate_total_price:
e.g. calculate_total_price: 2 apples, 3 oranges
Calculates the total price for the given fruits and quantities

Example session:

Question: What is the price of 2 bananas and 3 oranges?
Thought: I need to find the price of bananas and oranges first
Action: get_fruit_price: banana
PAUSE

You will be called again with this:

Observation: $1.2

You then output:

Action: get_fruit_price: orange
PAUSE

You will be called again with this:

Observation: $1.3

You then output:

Action: calculate_total_price: 2 bananas, 3 oranges
PAUSE

You will be called again with this:

Observation: $6.3

You then output:

Answer: The total price for 2 bananas and 3 oranges is $6.3
""".strip()

# Action parsing regex and known actions
action_re = re.compile(r'^Action: (\w+): (.*)$', re.MULTILINE)
known_actions = {
    "get_fruit_price": get_fruit_price,
    "calculate_total_price": calculate_total_price
}

print("Agent, tools, and log_event helper ready.")

Agent, tools, and log_event helper ready.


## Your Task: Enhance the Query Function with Advanced Logging

The basic `query_started` logging is already implemented. You need to add three enhancements:

1. **Tool execution timing** - Measure how long each tool takes and include `duration_ms` in the log
2. **Summary statistics** - Log `total_tools_called` and `tools_breakdown` when the query completes
3. **Error logging** - Log when the agent exceeds max turns without completing

Complete the TODO sections below.

In [7]:
def query(question: str, max_turns: int = 5) -> None:
    """Run agent loop with comprehensive logging.

    Logs these events:
    - 'query_started': Already implemented
    - 'tool_executed': Add duration_ms field (TODO 1)
    - 'query_completed': Add total_tools_called and tools_breakdown (TODO 2)
    - 'error_max_turns_exceeded': Log when agent hits turn limit (TODO 3)
    """
    # query_started logging (provided)
    log_event("query_started", {
        "question": question,
        "max_turns": max_turns
    })

    agent = Agent(system_prompt)
    next_prompt = question
    i = 0

    # Track tool usage for summary statistics
    tool_counts = {}  # e.g., {"get_fruit_price": 2, "calculate_total_price": 1}

    while i < max_turns:
        i += 1
        result = agent(next_prompt)
        print(f"\n=== Turn {i}/{max_turns} ===")
        print(result)

        actions = [
            action_re.match(a)
            for a in result.split('\n')
            if action_re.match(a)
        ]

        if actions:
            action, action_input = actions[0].groups()
            log_event("tool_planned", {"action": action, "action_input": action_input, "turn": i})
            if action not in known_actions:
                raise Exception(f"Unknown action: {action}: {action_input}")

            print(f"\n-- Running {action} with input: {action_input}")

            # TODO 1: Measure tool execution time and log tool_executed event
            #
            # Steps:
            # 1. Record start time using time.time()
            # 2. Execute the tool: observation = known_actions[action](action_input)
            # 3. Record end time using time.time()
            # 4. Calculate duration_ms = (end_time - start_time) * 1000
            # 5. Update tool_counts: increment count for this action
            # 6. Log "tool_executed" with: action, input, result, duration_ms
            #
            # Example log data:
            # {"action": "get_fruit_price", "input": "banana", "result": 1.2, "duration_ms": 52.3}

            start_time = time.time()
            observation = known_actions[action](action_input)  # Replace this line with your implementation
            end_time = time.time()
            duration_ms = (end_time - start_time) * 1000
            tool_counts[action] = tool_counts.get(action, 0) + 1
            log_event("tool_executed", { "action": action, "input": action_input, "result": observation, "duration_ms": duration_ms})

            print(f"Observation: {observation}")
            next_prompt = f"Observation: {observation}"
        else:
            # Agent provided an answer (no more actions)

            # TODO 2: Log query_completed with summary statistics
            #
            # Log "query_completed" with:
            # - total_turns: number of turns taken (i)
            # - total_tools_called: sum of all values in tool_counts
            # - tools_breakdown: the tool_counts dict
            #
            # Example log data:
            # {"total_turns": 4, "total_tools_called": 3, "tools_breakdown": {"get_fruit_price": 2, "calculate_total_price": 1}}

            total_tools_called = sum(tool_counts.values())
            log_event("query_completed",{"total_turns":i , "total_tools_called":total_tools_called, "tools_breakdown":tool_counts })
            return

    # If we reach here, agent exceeded max_turns without completing

    # TODO 3: Log error_max_turns_exceeded
    #
    # Log "error_max_turns_exceeded" with:
    # - question: the original question
    # - max_turns: the limit that was exceeded
    # - tools_called: total number of tools called (sum of tool_counts values)
    #
    # Example log data:
    # {"question": "What is...", "max_turns": 1, "tools_called": 1}

    log_event("error_max_turns_exceed",{"question": question, "max_turns":max_turns,"tools_called":sum(tool_counts.values())})

    print(f"\n*** Agent did not complete within {max_turns} turns ***")


print("Query function defined.")

Query function defined.


## Test Your Implementation

Run these tests to verify your logging enhancements work correctly.

In [10]:
# Test 1: Normal query
# Expected logs: query_started, tool_executed (with duration_ms), query_completed (with summary stats)
print("=" * 60)
print("TEST 1: Normal query with multiple tool calls")
print("=" * 60)
query("What is the price of 2 bananas and 3 oranges?")

2026-03-28 02:32:59 - INFO - {"timestamp": "2026-03-28T02:32:59.947122+00:00", "event": "query_started", "data": {"question": "What is the price of 2 bananas and 3 oranges?", "max_turns": 5}}
2026-03-28 02:32:59 - INFO - {"timestamp": "2026-03-28T02:32:59.951801+00:00", "event": "message_added", "data": {"role": "user", "content_preview": "What is the price of 2 bananas and 3 oranges?"}}
2026-03-28 02:32:59 - INFO - {"timestamp": "2026-03-28T02:32:59.957405+00:00", "event": "llm_input", "data": {"message_count": 2, "last_message_preview": "{'role': 'user', 'content': 'What is the price of 2 bananas and 3 oranges?'}"}}


TEST 1: Normal query with multiple tool calls


2026-03-28 02:33:04 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-28 02:33:04 - INFO - {"timestamp": "2026-03-28T02:33:04.418806+00:00", "event": "llm_output", "data": {"content_length": 101, "completion_preview": "Thought: I need to find the price of bananas and oranges first.\nAction: get_fruit_price: banana\nPAUSE", "tokens": {"prompt": 297, "completion": 25, "total": 322}, "model": "gpt-4o-2024-08-06"}}
2026-03-28 02:33:04 - INFO - {"timestamp": "2026-03-28T02:33:04.419814+00:00", "event": "message_added", "data": {"role": "assistant", "content_preview": "Thought: I need to find the price of bananas and oranges first.\nAction: get_fruit_price: banana\nPAUS..."}}
2026-03-28 02:33:04 - INFO - {"timestamp": "2026-03-28T02:33:04.420819+00:00", "event": "tool_planned", "data": {"action": "get_fruit_price", "action_input": "banana", "turn": 1}}
2026-03-28 02:33:04 - INFO - {"timestamp": "2026-03-28T02:33:04.472152+00:00", "event": "tool_


=== Turn 1/5 ===
Thought: I need to find the price of bananas and oranges first.
Action: get_fruit_price: banana
PAUSE

-- Running get_fruit_price with input: banana
Observation: 1.2


2026-03-28 02:33:08 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-28 02:33:08 - INFO - {"timestamp": "2026-03-28T02:33:08.202229+00:00", "event": "llm_output", "data": {"content_length": 37, "completion_preview": "Action: get_fruit_price: orange\nPAUSE", "tokens": {"prompt": 336, "completion": 11, "total": 347}, "model": "gpt-4o-2024-08-06"}}
2026-03-28 02:33:08 - INFO - {"timestamp": "2026-03-28T02:33:08.203039+00:00", "event": "message_added", "data": {"role": "assistant", "content_preview": "Action: get_fruit_price: orange\nPAUSE"}}
2026-03-28 02:33:08 - INFO - {"timestamp": "2026-03-28T02:33:08.205334+00:00", "event": "tool_planned", "data": {"action": "get_fruit_price", "action_input": "orange", "turn": 2}}
2026-03-28 02:33:08 - INFO - {"timestamp": "2026-03-28T02:33:08.256356+00:00", "event": "tool_executed", "data": {"action": "get_fruit_price", "input": "orange", "result": 1.3, "duration_ms": 50.12822151184082}}
2026-03-28 02:3


=== Turn 2/5 ===
Action: get_fruit_price: orange
PAUSE

-- Running get_fruit_price with input: orange
Observation: 1.3


2026-03-28 02:33:09 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-28 02:33:09 - INFO - {"timestamp": "2026-03-28T02:33:09.554190+00:00", "event": "llm_output", "data": {"content_length": 57, "completion_preview": "Action: calculate_total_price: 2 bananas, 3 oranges\nPAUSE", "tokens": {"prompt": 361, "completion": 16, "total": 377}, "model": "gpt-4o-2024-08-06"}}
2026-03-28 02:33:09 - INFO - {"timestamp": "2026-03-28T02:33:09.555198+00:00", "event": "message_added", "data": {"role": "assistant", "content_preview": "Action: calculate_total_price: 2 bananas, 3 oranges\nPAUSE"}}
2026-03-28 02:33:09 - INFO - {"timestamp": "2026-03-28T02:33:09.556056+00:00", "event": "tool_planned", "data": {"action": "calculate_total_price", "action_input": "2 bananas, 3 oranges", "turn": 3}}
2026-03-28 02:33:09 - INFO - {"timestamp": "2026-03-28T02:33:09.687327+00:00", "event": "tool_executed", "data": {"action": "calculate_total_price", "input": "2 banana


=== Turn 3/5 ===
Action: calculate_total_price: 2 bananas, 3 oranges
PAUSE

-- Running calculate_total_price with input: 2 bananas, 3 oranges
Observation: 6.300000000000001


2026-03-28 02:33:10 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-28 02:33:10 - INFO - {"timestamp": "2026-03-28T02:33:10.725343+00:00", "event": "llm_output", "data": {"content_length": 61, "completion_preview": "Answer: The total price for 2 bananas and 3 oranges is $6.30.", "tokens": {"prompt": 395, "completion": 19, "total": 414}, "model": "gpt-4o-2024-08-06"}}
2026-03-28 02:33:10 - INFO - {"timestamp": "2026-03-28T02:33:10.726572+00:00", "event": "message_added", "data": {"role": "assistant", "content_preview": "Answer: The total price for 2 bananas and 3 oranges is $6.30."}}
2026-03-28 02:33:10 - INFO - {"timestamp": "2026-03-28T02:33:10.728139+00:00", "event": "query_completed", "data": {"total_turns": 4, "total_tools_called": 3, "tools_breakdown": {"get_fruit_price": 2, "calculate_total_price": 1}}}



=== Turn 4/5 ===
Answer: The total price for 2 bananas and 3 oranges is $6.30.


In [11]:
# Test 2: Force max_turns exceeded
# Expected logs: query_started, tool_executed, error_max_turns_exceeded
print("=" * 60)
print("TEST 2: Max turns exceeded scenario")
print("=" * 60)
query("What is the price of apples and grapes?", max_turns=1)

2026-03-28 02:34:13 - INFO - {"timestamp": "2026-03-28T02:34:13.156978+00:00", "event": "query_started", "data": {"question": "What is the price of apples and grapes?", "max_turns": 1}}
2026-03-28 02:34:13 - INFO - {"timestamp": "2026-03-28T02:34:13.159434+00:00", "event": "message_added", "data": {"role": "user", "content_preview": "What is the price of apples and grapes?"}}
2026-03-28 02:34:13 - INFO - {"timestamp": "2026-03-28T02:34:13.160597+00:00", "event": "llm_input", "data": {"message_count": 2, "last_message_preview": "{'role': 'user', 'content': 'What is the price of apples and grapes?'}"}}


TEST 2: Max turns exceeded scenario


2026-03-28 02:34:14 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2026-03-28 02:34:14 - INFO - {"timestamp": "2026-03-28T02:34:14.545191+00:00", "event": "llm_output", "data": {"content_length": 118, "completion_preview": "Thought: I need to find the price of apples first, then find the price of grapes.\nAction: get_fruit_price: apple\nPAUSE", "tokens": {"prompt": 293, "completion": 30, "total": 323}, "model": "gpt-4o-2024-08-06"}}
2026-03-28 02:34:14 - INFO - {"timestamp": "2026-03-28T02:34:14.547110+00:00", "event": "message_added", "data": {"role": "assistant", "content_preview": "Thought: I need to find the price of apples first, then find the price of grapes.\nAction: get_fruit_..."}}
2026-03-28 02:34:14 - INFO - {"timestamp": "2026-03-28T02:34:14.548986+00:00", "event": "tool_planned", "data": {"action": "get_fruit_price", "action_input": "apple", "turn": 1}}
2026-03-28 02:34:14 - INFO - {"timestamp": "2026-03-28T02:34:14.601132+00:00", 


=== Turn 1/1 ===
Thought: I need to find the price of apples first, then find the price of grapes.
Action: get_fruit_price: apple
PAUSE

-- Running get_fruit_price with input: apple
Observation: 1.5

*** Agent did not complete within 1 turns ***


## Expected Output

**Test 1** should show:
- `query_started` with question and max_turns
- Multiple `tool_executed` events, each with a `duration_ms` field (e.g., 50-60ms)
- `query_completed` with `total_tools_called` (e.g., 3) and `tools_breakdown` (e.g., `{"get_fruit_price": 2, "calculate_total_price": 1}`)

**Test 2** should show:
- `query_started` with the question
- One `tool_executed` event with `duration_ms`
- `error_max_turns_exceeded` with question, max_turns=1, and tools_called count